# Ark Measurement DemoDemonstrates `caida_ai_ops.ark`'s capabilities end-to-end using its built-in **demo mode** (`MATTHEWPP_DEMO=1`), so this runs with no CAIDA credentials or Ark network access. Every response below carries an explicit `"DEMO MODE"` warning for that reason.Switch to **live mode** by removing the `MATTHEWPP_DEMO` env var and running inside CAIDA's Ark-enabled environment (see `etp-2026-overview/scamper-orientation.md` for access) — the function signatures and output shapes are identical either way; only the data source changes.

In [1]:
import os

os.environ.setdefault(
    "MATTHEWPP_DEMO", "1"
)  # synthetic fixtures; remove this line once you have real Ark access
import json

from caida_ai_ops import ark as m


def show(resp, limit=2000):
    text = json.dumps(resp, indent=2)
    print(text if len(text) <= limit else text[:limit] + "\n... (truncated)")

## 1. Discover vantage points

In [2]:
resp = m.list_vps(region="africa")
show(resp)

{
  "status": "ok",
  "function": "matthewpp.ark.vps.list_vps",
  "parameters": {
    "region": "africa",
    "country": null,
    "asn": null,
    "tag": null,
    "vp_ids": null,
    "near_lat_lon": null,
    "radius_km": 500.0,
    "ipv6_only": false,
    "active_only": true,
    "limit": null
  },
  "data": [
    {
      "vp_id": "ark-jnb-za",
      "hostname": "jnb-za.ark.caida.org",
      "asn": 36937,
      "org": "MTN SA",
      "country": "ZA",
      "region": "africa",
      "lat": -26.2041,
      "lon": 28.0473,
      "ipv4": true,
      "ipv6": true,
      "tags": [
        "residential"
      ],
      "last_heartbeat": "2026-08-19T18:46:47Z"
    },
    {
      "vp_id": "ark-lag-ng",
      "hostname": "lag-ng.ark.caida.org",
      "asn": 37146,
      "org": "MainOne",
      "country": "NG",
      "region": "africa",
      "lat": 6.5244,
      "lon": 3.3792,
      "ipv4": true,
      "ipv6": false,
      "tags": [],
      "last_heartbeat": "2026-08-19T18:46:47Z"
    }
  ],
 

## 2. PingThe canonical example from `ark-capabilities-and-modules.md`: *"take the nodes in Africa and ping google.com for 10 seconds."*

In [3]:
resp = m.ping("google.com", vp_filter={"region": "africa"}, duration_s=10)
show(resp)

{
  "status": "ok",
  "function": "matthewpp.ark.ping.ping",
  "parameters": {
    "target": "google.com",
    "vp_filter": {
      "region": "africa"
    },
    "duration_s": 10,
    "count": null,
    "interval_ms": 1000,
    "timeout_ms": 2000,
    "method": "icmp-echo",
    "parallel": true
  },
  "data": [
    {
      "vp_id": "ark-jnb-za",
      "target_ip": "google.com",
      "sent": 10,
      "received": 10,
      "loss_pct": 0.0,
      "rtt_min_ms": 70.0,
      "rtt_avg_ms": 71.35,
      "rtt_max_ms": 73.0,
      "rtt_stddev_ms": 1.25
    },
    {
      "vp_id": "ark-lag-ng",
      "target_ip": "google.com",
      "sent": 10,
      "received": 10,
      "loss_pct": 0.0,
      "rtt_min_ms": 130.0,
      "rtt_avg_ms": 131.35,
      "rtt_max_ms": 133.0,
      "rtt_stddev_ms": 1.25
    }
  ],
  "warnings": [
    "DEMO MODE (MATTHEWPP_DEMO=1): response built from synthetic fixture data, not live Ark results."
  ],
  "provenance": {
    "timestamp_utc": "2026-08-19T18:46:47Z",
    

## 3. Traceroute + flag a suspect detourThis mirrors the real incident `matthew-plus-plus.md` is built around: traffic unexpectedly routing through CAIDA's San Diego site (AS 7377) before reaching its destination. `compare_paths` runs a fresh traceroute and flags any VP whose path transits a suspect AS.

In [4]:
resp = m.compare_paths("example.com", vp_filter={"region": "east-asia"}, flag_transit_asn=7377)
show(resp)

{
  "status": "ok",
  "function": "matthewpp.ark.traceroute.compare_paths",
  "parameters": {
    "target": "example.com",
    "vp_filter": {
      "region": "east-asia"
    },
    "baseline_run_id": null,
    "flag_transit_asn": 7377,
    "flag_transit_ip_prefix": null
  },
  "data": {
    "run_id": "tr_20260819184647_2ae4db",
    "per_vp": [
      {
        "vp_id": "ark-nrt-jp",
        "path_changed": false,
        "hops_added": [],
        "hops_removed": [],
        "transits_flagged_asn": true,
        "transits_flagged_prefix": false
      },
      {
        "vp_id": "ark-sin-sg",
        "path_changed": false,
        "hops_added": [],
        "hops_removed": [],
        "transits_flagged_asn": true,
        "transits_flagged_prefix": false
      }
    ]
  },
  "warnings": [
    "DEMO MODE (MATTHEWPP_DEMO=1): response built from synthetic fixture data, not live Ark results."
  ],
  "provenance": {
    "timestamp_utc": "2026-08-19T18:46:47Z",
    "run_id": "tr_20260819184647_2

## 4. Multi-VP root-cause diagnosis`diagnose_path_anomaly` orchestrates VP sampling + `compare_paths` and returns **evidence** (percent of VPs affected, which regions) rather than a verdict — draw the conclusion yourself.

In [5]:
resp = m.diagnose_path_anomaly("example.com", suspect_asn=7377, vp_filter={"region": "east-asia"})
show(resp)

{
  "status": "ok",
  "function": "matthewpp.ark.diagnose.diagnose_path_anomaly",
  "parameters": {
    "target": "example.com",
    "suspect_asn": 7377,
    "suspect_ip_prefix": null,
    "vp_filter": {
      "region": "east-asia"
    },
    "sample_size": 15,
    "include_bgp_context": true
  },
  "data": {
    "pct_vps_affected": 100.0,
    "affected_regions": [
      "east-asia"
    ],
    "affected_vp_ids": [
      "ark-nrt-jp",
      "ark-sin-sg"
    ],
    "bgp_context": {
      "status": "unavailable",
      "asn": 7377,
      "reason": "BGP module not yet implemented"
    },
    "run_id": "tr_20260819184647_786b97"
  },
  "warnings": [
    "DEMO MODE (MATTHEWPP_DEMO=1): response built from synthetic fixture data, not live Ark results."
  ],
  "provenance": {
    "timestamp_utc": "2026-08-19T18:46:47Z"
  }
}

## 5. DNS divergenceDoes the DNS answer for a domain differ depending on which vantage point asks? (In this demo fixture, VPs in "oceania" see a different answer — a synthetic stand-in for real anycast/geo-DNS divergence.)

In [6]:
resp = m.dns_divergence_report("example.com")
show(resp)

{
  "status": "ok",
  "function": "matthewpp.ark.dns.dns_divergence_report",
  "parameters": {
    "qname": "example.com",
    "qtype": "A",
    "vp_filter": null,
    "resolver": "system"
  },
  "data": {
    "unique_answer_sets": 2,
    "majority_answer": [
      "198.51.100.9"
    ],
    "majority_vp_ids": [
      "ark-jnb-za",
      "ark-lag-ng",
      "ark-fra-de",
      "ark-lon-gb",
      "ark-nrt-jp",
      "ark-sin-sg",
      "ark-gru-br",
      "ark-sea-us",
      "ark-sdc-us"
    ],
    "minority_vps": [
      {
        "vp_id": "ark-syd-au",
        "answers": [
          "198.51.100.99"
        ]
      }
    ],
    "groups": [
      {
        "answers": [
          "198.51.100.9"
        ],
        "vp_ids": [
          "ark-jnb-za",
          "ark-lag-ng",
          "ark-fra-de",
          "ark-lon-gb",
          "ark-nrt-jp",
          "ark-sin-sg",
          "ark-gru-br",
          "ark-sea-us",
          "ark-sdc-us"
        ]
      },
      {
        "answers": [
    

## 6. Hard limits in action`caida_ai_ops.ark` enforces hard caps on every active-probing function so an agent can't accidentally (or on request) overload Ark or a probed target — see the module docstring for the full list. Exceeding one returns `status: "error"` with a `MeasurementLimitExceededError`, never a silently clamped/adjusted value.

In [7]:
# duration_s=500 exceeds the 60-second hard cap -- this returns status="error", not a clamped/slower ping
resp = m.ping("google.com", vp_filter={"region": "africa"}, duration_s=500)
show(resp)

{
  "status": "error",
  "function": "matthewpp.ark.ping.ping",
  "parameters": {
    "target": "google.com",
    "vp_filter": {
      "region": "africa"
    },
    "duration_s": 500,
    "count": null,
    "interval_ms": 1000,
    "timeout_ms": 2000,
    "method": "icmp-echo",
    "parallel": true
  },
  "data": null,
  "warnings": [],
  "provenance": {
    "timestamp_utc": "2026-08-19T18:46:47Z"
  },
  "error": {
    "type": "MeasurementLimitExceededError",
    "message": "duration_s must be between 0.001 and 60.0 (got 500)"
  }
}

## Next steps- **Live mode**: unset `MATTHEWPP_DEMO`, ensure the `scamper` package and Ark mux access are available (see `etp-2026-overview/scamper-orientation.md`), then re-run this notebook unchanged.- **MCP server**: `caida-ai-ops` exposes every one of these functions as an MCP tool for an agent to call directly — see `README.md`'s "MCP server" section for setup.- **Capability catalog**: `ark-capabilities-and-modules.md` documents every function's full signature, hard limits, and CLI equivalent.